In [1]:
# Test script 3

In [9]:
import os
import xarray as xr
import numpy as np
from utils.utils import adjust_longitude

In [10]:
OSDMA8_DIR = "/glade/work/awells/air_quality/CESM/ozone/OSDMA8/"
OBS_DIR = "/glade/work/awells/air_quality/O3_obs/"
scenario = "ARISE"
ens_num = 1

# Load data arrays
if scenario == "ARISE":
    dates = "2035-2068"
elif scenario == "SSP245":
    dates = "2020-2068"

osdma8_file = f"OSDMA8_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
osdma8_path = os.path.join(OSDMA8_DIR, osdma8_file)
# Convert from mol/mol to ppb
osdma8 = xr.open_dataarray(osdma8_path)*10**9

hist_file = "OSDMA8_CESM2_hist_01_1990-2008.nc"
hist_path = os.path.join(OSDMA8_DIR, hist_file)
# Convert from mol/mol to ppb
hist = xr.open_dataarray(hist_path)*10**9

obs_file = "Delang_BME_OSDMA8_1990_2017.nc"
obs_path = os.path.join(OBS_DIR, obs_file)
obs = xr.open_dataset(obs_path)["ozone"]

# Baseline years for fi_2000 and historical
base = slice("1990", "2008")
hist_base = hist.sel(year=base).mean("year")
obs_base = obs.sel(year=base).mean("year")
obs_base = obs_base.rename({'longitude': 'lon', 'latitude': 'lat'})

# Define the higher-resolution grid to match observations (0.1° x 0.1°)
new_lat = obs_base['lat']
new_lon = obs_base['lon']

# Calculate delta
delta_fi = adjust_longitude(osdma8 / hist_base)

# Interpolate to the new grid
ds_delta_fi = delta_fi.interp(lat=new_lat, lon=new_lon,
                              method='linear')

# Bias correct delta
bc_data = obs_base * ds_delta_fi

In [11]:
# === Define resolution and bounds ===
lat_res = 1 / 10  # 0.1 degrees
lon_res = 1 / 10

# Generate full latitude and longitude ranges
lat_full = np.arange(-90 + lat_res / 2, 90, lat_res)
lon_full = np.arange(-180 + lon_res / 2, 180, lon_res)

# Confirm lengths match 0.1 x 0.1 grid
print((f"Longitude points should be 3600: {len(lon_full)}, "
       f"Latitude points should be 1800: {len(lat_full)}"))

# === Create new empty DataArray with NaNs ===
shape = (len(lat_full), len(lon_full), len(bc_data.year))
bc_full = xr.DataArray(
    data=np.full(shape, np.nan),
    coords={"lat": lat_full, "lon": lon_full, "year": bc_data.year.values},
    dims=["lat", "lon", "year"],
    name="OSDMA8 Bias Corrected"
)

Longitude points should be 3600: 3600, Latitude points should be 1800: 1800


In [18]:
# === Add OSDMA8 data to empty DataArray ===
# Adjust indices to match (with small tolerance) e.g., max 0.01° distance
bc_nearest = bc_data.reindex_like(bc_full, method="nearest", tolerance=0.01)

# Add data to bc_full
mask = xr.where(bc_full.isnull(), True, False)
bc_full = xr.where(mask, bc_nearest, bc_full)

In [21]:
bc_full = bc_full.drop_vars("time")
bc_full.to_netcdf("/glade/work/awells/air_quality/CESM/ozone/OSDMA8_BC/test.nc")